In [1]:
import sys
from pathlib import Path
import geopandas as gpd
from datetime import datetime, timezone
import pandas as pd
from tqdm import tqdm
import shutil

# Find the repository root.
PROJECT_ROOT = next(
    path
    for path in (Path.cwd(), *Path.cwd().parents)
    if (path / "src").is_dir()
)

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.config import load_config, photo_paths
from src.prep import (
    download_photo,
    get_image_metadata,
    calculate_ground_extent,
    ground_to_mercator_extent,
    update_tracker,
)

cfg = load_config()

In [2]:
INDEX_PATH = cfg["paths"]["index"]
print("Source index:", INDEX_PATH)

Source index: C:\Users\eu01251115\LocalProjects\batch_georef\data\main_index_blank.gpkg


In [3]:
# Load the main index.
LAYER = "photos"
index_gdf = gpd.read_file(INDEX_PATH, layer=LAYER)

### Examine the available years and projects
The second cell below displays unique combos between flight codes and 

In [4]:
# examine all years in the county

county_gdf = index_gdf.loc[index_gdf["county"] == cfg["county"]]

for year in sorted(county_gdf["year"].dropna().unique()):
    print(year)

1937
1938
1940
1947
1951
1953
1955
1956
1957
1960
1962
1963
1964
1967
1969
1971


In [6]:
# Examine unique year + flight code combinations for the county
county_gdf = index_gdf.loc[index_gdf["county"] == cfg["county"]]

flights = (
    county_gdf[["year", "flight_code", "county"]]
    .dropna()
    .drop_duplicates()
    .sort_values(["year", "flight_code", "county"])
)

display(flights)

,year,flight_code,county
13511,1937,Carver,Carver
13788,1937,Hennepin,Carver
13773,1937,Scott,Carver
13804,1938,bjm,Carver
13809,1940,Carver,Carver
13886,1940,McLeod,Carver
13894,1940,Scott,Carver
13897,1947,dv,Carver
14059,1951,Carver,Carver
14173,1951,Scott,Carver


### Manually determining ground scale:

- Visit the [MHAPO Air Photo Holdings page](https://www.lib.umn.edu/collections/borchert/aerial-photography-and-remote-sensed-imagery). 
- Select your chosen county to open a PDF with metadata. 
- Compare this to the list above to determine the ground scale for your target year or years. Currently, this tool supports processing multiple years at the same time, but only if they share the same ground scale. 
- **Choose a year or multiple years with the same scale**
- Specify the year(s) and scale for your current batch in config.yml.
- The batch will be named for the county, years, and scale. The photos and the folders they are in will be named for the county and year. 

#### Once you have configured your year(s) and scale in config.yml, restart the kernel and start again from the beginning.


In [8]:
# Select records for the configured county and years.
mask = (
    index_gdf["county"].str.casefold().eq(
        cfg["county"].casefold()
    )
    & index_gdf["year"].isin(cfg["years"])
)

In [9]:
batch_gdf = index_gdf.loc[mask].copy()

In [11]:
TRACKER_PATH = cfg["paths"]["tracker"]
LAYER = "photos"
print("Path for tracker:", TRACKER_PATH)
print("County:", cfg["county"])
print("Years:", cfg["years"])
print("Scale:", cfg["scale"])

Path for tracker: C:\Users\eu01251115\LocalProjects\batch_georef\data\batches\carver_1953_60000_tracker.gpkg
County: Carver
Years: [1953]
Scale: 60000


## Create a tracker table

In [12]:
if batch_gdf.empty:
    raise ValueError("No photographs match this batch configuration.")

if (
    batch_gdf["cty_year_id"].isna().any()
    or not batch_gdf["cty_year_id"].is_unique
):
    raise ValueError("Batch contains missing or duplicate photo IDs.")

In [13]:
# check length and confirm CRS
print(f"Selected {len(batch_gdf)} photographs")
print("CRS:", batch_gdf.crs)

Selected 8 photographs
CRS: EPSG:3857


In [14]:
# Assign the configured scale to all selected photographs.
batch_gdf["scale"] = cfg["scale"]

In [15]:
# preview it
display(
    batch_gdf[
        ["cty_year_id", "county", "year", "scale", "url"]
    ].head()
)

,cty_year_id,county,year,scale,url
14174,carver_1953_wn-5m-125,Carver,1953,60000,https://geo.lib.umn.edu/hennepin_county/y1953/...
14175,carver_1953_wn-5m-90,Carver,1953,60000,https://geo.lib.umn.edu/hennepin_county/y1953/...
14176,carver_1953_wn-5m-88,Carver,1953,60000,https://geo.lib.umn.edu/hennepin_county/y1953/...
14177,carver_1953_wn-5m-37,Carver,1953,60000,https://geo.lib.umn.edu/hennepin_county/y1953/...
14178,carver_1953_wn-5m-39,Carver,1953,60000,https://geo.lib.umn.edu/hennepin_county/y1953/...


In [16]:
batch_gdf.columns

Index(['cty_year_id', 'status', 'county', 'year', 'photo_id', 'ground_width',
       'ground_height', 'wm_width', 'wm_height', 'scale', 'dpi_x', 'dpi_y',
       'width_px', 'height_px', 'worldfile_json', 'url', 'flight_code',
       'source_id', 'date_prepared', 'date_georef', 'gcp_count', 'confidence',
       'error', 'latitude', 'geometry'],
      dtype='str')

In [17]:
if TRACKER_PATH.exists():
    raise FileExistsError(
        f"Batch tracker already exists: {TRACKER_PATH}"
    )

TRACKER_PATH.parent.mkdir(parents=True, exist_ok=True)

batch_gdf.to_file(
    TRACKER_PATH,
    layer=LAYER,
    driver="GPKG",
)

print(f"Created batch tracker: {TRACKER_PATH}")

Created batch tracker: C:\Users\eu01251115\LocalProjects\batch_georef\data\batches\carver_1953_60000_tracker.gpkg


## Download JPEGS and calculate extents

In [18]:
# Load the batch tracker for preparation
tracker_gdf = gpd.read_file(TRACKER_PATH, layer=LAYER)

print(f"Batch: {cfg['batch_name']}")
print(f"Photographs: {len(tracker_gdf)}")

display(tracker_gdf.head())

Batch: carver_1953_60000
Photographs: 8


,cty_year_id,status,county,year,photo_id,ground_width,ground_height,wm_width,wm_height,scale,...,url,flight_code,source_id,date_prepared,date_georef,gcp_count,confidence,error,latitude,geometry
0,carver_1953_wn-5m-125,None,Carver,1953,wn-5m-125,NaN,NaN,NaN,NaN,60000,...,https://geo.lib.umn.edu/hennepin_county/y1953/...,Hennepin,93480,None,None,None,None,None,44.819129,POINT (-10411632.826 5593091.851)
1,carver_1953_wn-5m-90,None,Carver,1953,wn-5m-90,NaN,NaN,NaN,NaN,60000,...,https://geo.lib.umn.edu/hennepin_county/y1953/...,Hennepin,93481,None,None,None,None,None,44.847898,POINT (-10415163.435 5597607.839)
2,carver_1953_wn-5m-88,None,Carver,1953,wn-5m-88,NaN,NaN,NaN,NaN,60000,...,https://geo.lib.umn.edu/hennepin_county/y1953/...,Hennepin,93494,None,None,None,None,None,44.877091,POINT (-10412837.303 5602192.689)
3,carver_1953_wn-5m-37,None,Carver,1953,wn-5m-37,NaN,NaN,NaN,NaN,60000,...,https://geo.lib.umn.edu/hennepin_county/y1953/...,Hennepin,93501,None,None,None,None,None,44.906789,POINT (-10443421.331 5606859.239)
4,carver_1953_wn-5m-39,None,Carver,1953,wn-5m-39,NaN,NaN,NaN,NaN,60000,...,https://geo.lib.umn.edu/hennepin_county/y1953/...,Hennepin,93502,None,None,None,None,None,44.906888,POINT (-10438578.376 5606874.799)


In [19]:
# Download original JPEGs.
download_results = []

for _, photo in tqdm(
    tracker_gdf.iterrows(),
    total=len(tracker_gdf),
    desc="Downloading photographs",
):
    photo_id = str(photo["cty_year_id"])
    jpeg_path = photo_paths(photo, cfg)["jpeg"]

    try:
        download_photo(photo["url"], jpeg_path)

        download_results.append({
            "cty_year_id": photo_id,
            "jpeg_path": str(jpeg_path),
            "error": None,
        })

    except Exception as exc:
        print(f"FAILED: {photo_id}: {exc}")

        download_results.append({
            "cty_year_id": photo_id,
            "jpeg_path": str(jpeg_path),
            "error": str(exc),
        })

download_df = pd.DataFrame(download_results)

print(f"Successful downloads: {download_df['error'].isna().sum()}")
print(f"Failed downloads: {download_df['error'].notna().sum()}")

display(download_df.head())

Successful downloads: 8
Failed downloads: 0


,cty_year_id,jpeg_path,error
0,carver_1953_wn-5m-125,C:\Users\eu01251115\LocalProjects\batch_georef...,None
1,carver_1953_wn-5m-90,C:\Users\eu01251115\LocalProjects\batch_georef...,None
2,carver_1953_wn-5m-88,C:\Users\eu01251115\LocalProjects\batch_georef...,None
3,carver_1953_wn-5m-37,C:\Users\eu01251115\LocalProjects\batch_georef...,None
4,carver_1953_wn-5m-39,C:\Users\eu01251115\LocalProjects\batch_georef...,None


In [20]:
# Collect preparation results for each photo.
results = []

for _, photo in tqdm(
    tracker_gdf.iterrows(),
    total=len(tracker_gdf),
    desc="Calculating extents",
):
    photo_id = photo["cty_year_id"]
    image_path = photo_paths(photo, cfg)["jpeg"]

    result = {
        "cty_year_id": photo_id,
        "width_px": None,
        "height_px": None,
        "dpi_x": None,
        "dpi_y": None,
        "ground_width": None,
        "ground_height": None,
        "wm_width": None,
        "wm_height": None,
        "date_prepared": None,
        "error": None,
    }

    try:
        # Read metadata from the existing JPEG.
        metadata = get_image_metadata(image_path)
        result.update(metadata)

        # Calculate ground dimensions in meters.
        result["ground_width"] = calculate_ground_extent(
            pixels=metadata["width_px"],
            dpi=metadata["dpi_x"],
            scale=photo["scale"],
        )

        result["ground_height"] = calculate_ground_extent(
            pixels=metadata["height_px"],
            dpi=metadata["dpi_y"],
            scale=photo["scale"],
        )

        # Convert dimensions to Web Mercator.
        result["wm_width"] = ground_to_mercator_extent(
            ground_extent_m=result["ground_width"],
            latitude=photo["latitude"],
        )

        result["wm_height"] = ground_to_mercator_extent(
            ground_extent_m=result["ground_height"],
            latitude=photo["latitude"],
        )

        # Mark preparation complete only after successful calculations.
        result["date_prepared"] = datetime.now(
            timezone.utc
        ).date()

    except Exception as e:
        result["error"] = str(e)
        print(f"FAILED: {photo_id}: {e}")

    results.append(result)

# Convert results to a DataFrame for inspection.
results_df = pd.DataFrame(results)

print(f"Successful: {results_df['error'].isna().sum()}")
print(f"Failed: {results_df['error'].notna().sum()}")

display(results_df.head())

Calculating extents: 100%|██████████| 8/8 [00:00<00:00, 1041.58it/s]

Successful: 8
Failed: 0


,cty_year_id,width_px,height_px,dpi_x,dpi_y,ground_width,ground_height,wm_width,wm_height,date_prepared,error
0,carver_1953_wn-5m-125,5486,5456,600,600,13934.44,13858.24,19644.358813,19536.934321,2026-11-18,None
1,carver_1953_wn-5m-90,5446,5448,600,600,13832.84,13837.92,19510.863631,19518.028840,2026-11-18,None
2,carver_1953_wn-5m-88,5438,5440,600,600,13812.52,13817.60,19492.084224,19499.253067,2026-11-18,None
3,carver_1953_wn-5m-37,5430,5464,600,600,13792.20,13878.56,19473.461893,19595.395172,2026-11-18,None
4,carver_1953_wn-5m-39,5486,5456,600,600,13934.44,13858.24,19674.327061,19566.738688,2026-11-18,None


In [21]:
# Update the tracker table
updated_count = update_tracker(
    TRACKER_PATH,
    results_df,
    layer=LAYER,
)

print(f"Updated {updated_count} records.")

Updated 8 records.


### Final load & preview

In [22]:
tracker_gdf = gpd.read_file(TRACKER_PATH, layer=LAYER)

tracker_gdf

,cty_year_id,status,county,year,photo_id,ground_width,ground_height,wm_width,wm_height,scale,...,url,flight_code,source_id,date_prepared,date_georef,gcp_count,confidence,error,latitude,geometry
0,carver_1953_wn-5m-125,prepared,Carver,1953,wn-5m-125,13934.44,13858.24,19644.358813,19536.934321,60000,...,https://geo.lib.umn.edu/hennepin_county/y1953/...,Hennepin,93480,2026-11-18,None,None,None,None,44.819129,POINT (-10411632.826 5593091.851)
1,carver_1953_wn-5m-90,prepared,Carver,1953,wn-5m-90,13832.84,13837.92,19510.863631,19518.028840,60000,...,https://geo.lib.umn.edu/hennepin_county/y1953/...,Hennepin,93481,2026-11-18,None,None,None,None,44.847898,POINT (-10415163.435 5597607.839)
2,carver_1953_wn-5m-88,prepared,Carver,1953,wn-5m-88,13812.52,13817.60,19492.084224,19499.253067,60000,...,https://geo.lib.umn.edu/hennepin_county/y1953/...,Hennepin,93494,2026-11-18,None,None,None,None,44.877091,POINT (-10412837.303 5602192.689)
3,carver_1953_wn-5m-37,prepared,Carver,1953,wn-5m-37,13792.20,13878.56,19473.461893,19595.395172,60000,...,https://geo.lib.umn.edu/hennepin_county/y1953/...,Hennepin,93501,2026-11-18,None,None,None,None,44.906789,POINT (-10443421.331 5606859.239)
4,carver_1953_wn-5m-39,prepared,Carver,1953,wn-5m-39,13934.44,13858.24,19674.327061,19566.738688,60000,...,https://geo.lib.umn.edu/hennepin_county/y1953/...,Hennepin,93502,2026-11-18,None,None,None,None,44.906888,POINT (-10438578.376 5606874.799)
5,carver_1953_wn-5m-34,prepared,Carver,1953,wn-5m-34,13792.20,13837.92,19483.789452,19548.376600,60000,...,https://geo.lib.umn.edu/hennepin_county/y1953/...,Hennepin,93532,2026-11-18,None,None,None,None,44.937250,POINT (-10442206.056 5611648.186)
6,carver_1953_wn-4m-177,prepared,Carver,1953,wn-4m-177,13873.48,13837.92,19607.741723,19557.483871,60000,...,https://geo.lib.umn.edu/hennepin_county/y1953/...,Hennepin,93533,2026-11-18,None,None,None,None,44.963983,POINT (-10443208.265 5615853.128)
7,carver_1953_wn-4m-179,prepared,Carver,1953,wn-4m-179,13832.84,13858.24,19550.290547,19586.188987,60000,...,https://geo.lib.umn.edu/hennepin_county/y1953/...,Hennepin,93534,2026-11-18,None,None,None,None,44.963943,POINT (-10438290.282 5615846.834)
